# Vision fine-tune: visual 2-step transformation-error diagnoser

QLoRA fine-tune **and** baseline for the vision path. The model is shown a coordinate-grid image containing three polygons — **RED** pre-image, **GREEN** dashed correct image, **BLUE** student answer — and must return a JSON diagnosis of the student's mistake.

**Base model:** `unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit` (Qwen3-VL 4B, fits a free T4 in 4-bit).

**Data:** generated by the `transform_diagnosis` package — 8-label taxonomy, deterministic per-label hints, and a held-out rotation∘reflection **OOD** split. Training conversations are pre-built in `*_chat.jsonl` (each line = image + instruction → target JSON). This notebook loads them, swaps each image path for a decoded PIL image, and trains with `UnslothVisionDataCollator`.

**Target JSON:** `{"label", "correct_transform", "hint"}` — one of 8 labels, the intended two-step transform (RED→GREEN), and a Socratic hint. The instruction (color key + schema) is embedded in every record's user turn, so it travels with the data.

**Runtime:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

In [ ]:
# Confirm a GPU is attached (expect a T4 on free Colab).
!nvidia-smi

In [ ]:
# Install Unsloth (includes the vision fine-tuning stack). Output is visible so errors show.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
# If pip upgrades torch, do Runtime > Restart session once, then run the cells below (skip this one).

In [ ]:
from unsloth import FastVisionModel
import torch

MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)  # enable Unsloth's faster inference path
print("loaded:", MODEL_NAME)

In [ ]:
# --- Dataset location ---------------------------------------------------------------------
# Get the `transform_diagnosis_data/` folder (the *_chat.jsonl files + the renders/ dir) onto
# this Colab session. Two easy options:
#   (a) zip it locally, drag the zip into the Files pane, then:
#         !unzip -q transform_diagnosis_data.zip -d /content
#   (b) mount Drive:
#         from google.colab import drive; drive.mount('/content/drive')
#         DATA_DIR = '/content/drive/MyDrive/transform_diagnosis_data'
import os

DATA_DIR = "transform_diagnosis_data"          # <-- edit if your folder lives elsewhere
assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR!r} (upload the dataset folder)"
print("data dir:", DATA_DIR)
print("chat splits:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith("_chat.jsonl")))

In [ ]:
# --- Make the transform_diagnosis package importable, then load the eval harness ----------
# The eval harness (scoring + metrics) lives in the repo's `transform_diagnosis/` package.
# It's pure Python (no torch), so import it once and reuse it for both base and tuned scoring.
# Get the package onto Colab by ONE of:
#   * !git clone <your-repo-url>          then set CODE_DIR to the repo root
#   * drag the `transform_diagnosis/` folder into the Files pane   (CODE_DIR = "/content")
import sys

CODE_DIR = "."   # <-- the parent dir that CONTAINS the `transform_diagnosis/` package
for cand in (CODE_DIR, "/content", "/content/SLM", ".."):
    if os.path.isdir(os.path.join(cand, "transform_diagnosis")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break
else:
    raise FileNotFoundError(
        "Couldn't find the `transform_diagnosis` package. Clone the repo or upload the "
        "folder, then set CODE_DIR to its parent directory."
    )

from transform_diagnosis import eval as ev, transform_core as tc

LABELS = tc.DIAGNOSIS_LABELS      # single source of truth for the 8-label vocab (no drift)
print("eval harness loaded; labels:", LABELS)

In [ ]:
# --- Load chat conversations and decode images to PIL -------------------------------------
# Each line of <split>_chat.jsonl is {"id", "split", "messages": [user, assistant]}. The user
# turn references its image by path; UnslothVisionDataCollator needs a decoded PIL image, so we
# swap the path for Image.open(...).convert("RGB") in place.
import json
from PIL import Image

def load_split(split):
    rows = []
    with open(os.path.join(DATA_DIR, f"{split}_chat.jsonl")) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            for msg in rec["messages"]:
                for part in msg["content"]:
                    if part.get("type") == "image" and isinstance(part.get("image"), str):
                        part["image"] = Image.open(
                            os.path.join(DATA_DIR, part["image"])
                        ).convert("RGB")
            rows.append(rec)
    return rows

train_rows = load_split("train")
val_rows   = load_split("val")
test_rows  = load_split("test")
ood_rows   = load_split("ood")
print({s: len(r) for s, r in
       [("train", train_rows), ("val", val_rows), ("test", test_rows), ("ood", ood_rows)]})

def ground_truth(rec):
    """The record's own assistant turn is the gold target JSON."""
    return json.loads(rec["messages"][1]["content"][0]["text"])

print("example gold target:", ground_truth(train_rows[0]))

In [ ]:
# --- Format-fit check: does the Qwen3-VL chat template accept our conversations? ----------
# Render one conversation through the tokenizer's chat template. You should see the Qwen vision
# placeholders (e.g. <|vision_start|> ... <|vision_end|>) where the image goes, and the JSON
# target as the assistant turn. If this prints without error, the format is fit for the model.
proof = tokenizer.apply_chat_template(train_rows[0]["messages"], tokenize=False)
print(proof[:1400])

# Unsloth's vision collator consumes a plain list of {"messages": [...]} dicts (images as PIL).
# Keep only "messages" for training.
train_dataset = [{"messages": r["messages"]} for r in train_rows]
print("\ntrain examples:", len(train_dataset))

In [ ]:
# --- Inference + programmatic scoring via the eval harness --------------------------------
# run_model: generate the raw diagnosis string for one record's user turn (GPU).
# Scoring is done by the harness (transform_diagnosis/eval.py): each output -> yes/no on
# parse_ok / label_ok / transform_ok / hint_ok, plus balanced accuracy and an 8x8 confusion.
# We grade against the FULL raw record (test.jsonl / ood.jsonl), because hint-token checking
# needs `student_transform`, which the *_chat.jsonl rows do not carry.

EVAL_N = None   # None = score the FULL frozen split (slow on a T4: ~thousands of gens).
                # Set an int (e.g. 300) for a faster fixed sample — keep it identical across
                # base and tuned so the comparison stays honest.

def run_model(user_message, image, max_new_tokens=256):
    """Generate the model's raw diagnosis string for one record's user turn."""
    input_text = tokenizer.apply_chat_template([user_message], add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def load_raw(split):
    """Full oracle records for a split, keyed by id (carry correct_transform,
    student_transform, hint — everything score_record needs)."""
    by_id = {}
    with open(os.path.join(DATA_DIR, f"{split}.jsonl")) as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                by_id[r["id"]] = r
    return by_id

def run_eval(tag, chat_rows, raw_by_id, n=EVAL_N):
    """Score the model over chat_rows against the oracle, save results, return the aggregate.

    Writes results_<tag>.json (metrics) and records_<tag>.jsonl (one scored row per record,
    with raw_model_output + failure_reason — the file you read for error analysis).
    """
    rows = chat_rows if n is None else chat_rows[:n]
    scored = []
    for row in rows:
        user_msg = row["messages"][0]
        image = next(p["image"] for p in user_msg["content"] if p.get("type") == "image")
        text = run_model(user_msg, image)
        scored.append(ev.score_record(text, raw_by_id[row["id"]]))
    agg = ev.aggregate(scored)
    ev.save_results(agg, f"results_{tag}.json", scored, f"records_{tag}.jsonl")
    print(f"[{tag}] n={agg['n']} label_acc={agg['label_accuracy']:.3f} "
          f"balanced_acc={agg['balanced_accuracy']:.3f} parse={agg['parse_rate']:.3f} "
          f"transform={agg['transform_match_rate']:.3f} hint={agg['hint_match_rate']:.3f}")
    return agg

# Oracle records for the frozen evaluation splits (+ val, used only for Day-4 iteration).
raw_test = load_raw("test")
raw_ood  = load_raw("ood")
raw_val  = load_raw("val")
print("raw records:", {"test": len(raw_test), "ood": len(raw_ood), "val": len(raw_val)})

In [ ]:
# --- Baseline (before fine-tune) on the FROZEN test + ood splits --------------------------
# The number the fine-tune must beat. test/ood are frozen: scored here, and exactly once more
# after training. ALL iteration/failure-reading happens on `val` — never tune against test.
FastVisionModel.for_inference(model)
base_test = run_eval("base_test", test_rows, raw_test)
base_ood  = run_eval("base_ood",  ood_rows,  raw_ood)

In [ ]:
# --- QLoRA vision fine-tune ---------------------------------------------------------------
from unsloth import FastVisionModel, is_bf16_supported
try:
    from unsloth import UnslothVisionDataCollator
except Exception:
    from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    random_state=3407, use_rslora=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,               # one pass over ~9.6k images; raise if time allows
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        # vision-specific: keep raw messages and let the collator build the batches
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=1,
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
# --- After fine-tune: re-score the SAME frozen test + ood splits --------------------------
# OOD = the two rotation-reflection compositions never seen in training; it probes
# compositional generalization. Its slice is unbalanced, so read balanced_acc there.
FastVisionModel.for_inference(model)
tuned_test = run_eval("tuned_test", test_rows, raw_test)
tuned_ood  = run_eval("tuned_ood",  ood_rows,  raw_ood)

In [ ]:
# --- Deliverable: base-vs-tuned tables + confusion + failure breakdown --------------------
import collections

print("== IN-DISTRIBUTION TEST (base vs tuned) ==")
print(ev.format_table(base_test, tuned_test))
print("\n== OOD / held-out compositions (base vs tuned) ==")
print(ev.format_table(base_ood, tuned_ood))

print("\n== Tuned TEST confusion (true rows x predicted cols; PF = parse fail) ==")
print(ev.format_confusion(tuned_test))
print("\n== Tuned OOD confusion ==")
print(ev.format_confusion(tuned_ood))

print("\n== Per-label recall (tuned test) ==")
for lab, rec in tuned_test["per_label_recall"].items():
    print(f"  {lab:34s} {'--' if rec is None else f'{rec:.3f}'}")

# Error analysis: the worst failure modes, read straight from the saved per-record rows.
fails = collections.Counter()
with open("records_tuned_test.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if r["failure_reason"]:
            kind = r["failure_reason"].split(":")[0]      # collapse wrong_label:X->Y
            fails[(r["true_label"], kind)] += 1
print("\n== Top tuned-test failure modes (true_label, kind) ==")
for (lab, kind), c in fails.most_common(12):
    print(f"  {c:4d}  {lab:34s} {kind}")
print("\nSaved: results_/records_ {base,tuned}_{test,ood}. Do error analysis + data fixes on"
      " `val` (raw_val), retrain, then re-run the two eval cells above for the final numbers.")

## Reading the results

The harness grades every output with four **binary checks** (yes/no), reported as the % that pass:

- **`parse_rate`** — output was valid JSON with a known label. Parse failures count as wrong everywhere else.
- **`label_acc`** — predicted label == oracle label (exact).
- **`balanced_acc`** — mean per-label recall over the labels *present* in the split. This is the honest headline on the **unbalanced OOD** slice; prefer it over raw accuracy there.
- **`transform_match`** — predicted `correct_transform` composes to the same net map as the oracle (wording-invariant, via `transform_core`).
- **`hint_match`** — the hint names the right error (expected tokens present, from `hints.expected_hint_tokens`) **and** leaks no coordinates it wasn't sanctioned to state. Deterministic — no LLM judge.

Per-run artifacts: `results_<tag>.json` (aggregate) and `records_<tag>.jsonl` (one scored row per record with `raw_model_output` + `failure_reason`) for `tag ∈ {base,tuned}_{test,ood}`.

## Split discipline (don't break this)

`test` + `ood` are **frozen** — scored exactly twice (baseline, final tuned). All iteration and failure-reading happens on **`val`** (`raw_val` is loaded for exactly this). If you patch data to fix things you saw in `test`, the improvement is fiction.

## Next steps
1. **Iterate on `val`:** read `records_tuned_test`-style rows for a `val` run, find the worst `(label, failure)` mode, **fix it in the data** (regenerate that slice), retrain, re-check on `val`. Touch `test`/`ood` only for the final re-run.
2. **Optional — deepen:** log run scores to **Langfuse**; add an LLM-as-judge for hint *fluency* (quality, beyond the deterministic token check); compare base vs. tuned vs. a **prompted frontier VLM** on the same frozen splits; build an **adversarial OOD** set.
3. **Sweep** LoRA rank / epochs; save adapters with `model.save_pretrained("lora_adapters")` (and push to the Hub if desired).